In [12]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import lightgbm as lgb

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [13]:
# ----------------------------------
# 1. 데이터 로드 및 정렬
# ----------------------------------

df = pd.read_csv('extracted_data.csv')
df = df.sort_values(by=['id', 'year']).reset_index(drop=True)

print(f'전체 행: {len(df):,}  |  고유 고객: {df["id"].nunique():,}')

전체 행: 147,915  |  고유 고객: 19,840


In [14]:
# ----------------------------------
# 2. 파생변수 생성
# ----------------------------------

# 통신사 변경 이력
df['prev_provider']    = df.groupby('id')['provider'].shift(1)
df['provider_changed'] = (df['provider'] != df['prev_provider']).astype(int)
df['total_changes']    = df.groupby('id')['provider_changed'].cumsum()

# 전년 대비 통신비 변화율
df['prev_cost']         = df.groupby('id')['monthly_total_cost'].shift(1)
df['cost_change_rate']  = (df['monthly_total_cost'] - df['prev_cost']) / (df['prev_cost'] + 1)

# 가입 기간
df['tenure'] = df.groupby('id').cumcount() + 1

print('파생변수 생성 완료')
print(f'  - provider_changed : 전년 대비 통신사 변경 여부')
print(f'  - total_changes    : 누적 통신사 변경 횟수')
print(f'  - cost_change_rate : 전년 대비 통신비 변화율')
print(f'  - tenure           : 가입 기간 (관측 연수)')

파생변수 생성 완료
  - provider_changed : 전년 대비 통신사 변경 여부
  - total_changes    : 누적 통신사 변경 횟수
  - cost_change_rate : 전년 대비 통신비 변화율
  - tenure           : 가입 기간 (관측 연수)


In [15]:
# ----------------------------------
# 3. next_provider 생성
# ----------------------------------

df['next_provider'] = df.groupby('id')['provider'].shift(-1)

provider_map = {1.0: 'SKT', 2.0: 'KT', 3.0: 'LGU+'}

print('[next_provider 분포]')
print(df['next_provider'].value_counts(dropna=False))

[next_provider 분포]
next_provider
1.0    59124
2.0    33662
NaN    27100
3.0    26318
4.0     1606
5.0      105
Name: count, dtype: int64


In [16]:
# ----------------------------------
# 4. 실제 전환 이탈 고객 추출
# ----------------------------------

switching_df = df[
    (df['churn'] == 1.0) &
    (df['next_provider'].notna()) &
    (df['provider'].notna()) &
    (df['provider'].isin([1.0, 2.0, 3.0])) &
    (df['next_provider'].isin([1.0, 2.0, 3.0])) &
    (df['provider'] != df['next_provider'])
].copy()

print(f'대상 고객 수: {len(switching_df):,}명')
print('\n[현재 통신사 분포]')
print(switching_df['provider'].map(provider_map).value_counts())
print('\n[이동 통신사 분포]')
print(switching_df['next_provider'].map(provider_map).value_counts())

대상 고객 수: 42,320명

[현재 통신사 분포]
provider
SKT     16974
KT      13875
LGU+    11471
Name: count, dtype: int64

[이동 통신사 분포]
next_provider
SKT     16637
KT      13702
LGU+    11981
Name: count, dtype: int64


In [17]:
# ----------------------------------
# 5. 전처리
# ----------------------------------

switching_df['monthly_installment'] = switching_df['monthly_installment'].fillna(0)
switching_df['monthly_total_cost']  = switching_df['monthly_total_cost'].fillna(switching_df['monthly_total_cost'].median())
switching_df['is_mobile_bundled']   = switching_df['is_mobile_bundled'].fillna(0)
switching_df['household_size']      = switching_df['household_size'].fillna(switching_df['household_size'].median())
switching_df['prev_cost']           = switching_df['prev_cost'].fillna(switching_df['monthly_total_cost'])
switching_df['cost_change_rate']    = switching_df['cost_change_rate'].fillna(0)
switching_df['provider_changed']    = switching_df['provider_changed'].fillna(0)
switching_df['total_changes']       = switching_df['total_changes'].fillna(0)

# 피처 목록
features = [
    # 기존 피처
    'age', 'gender', 'income', 'school', 'area', 'job', 'marriage',
    'monthly_total_cost', 'monthly_installment', 'cost_payer',
    'provider', 'household_size', 'is_mobile_bundled',
    # 신규 파생변수
    'provider_changed', 'total_changes', 'cost_change_rate', 'tenure'
]

X = switching_df[features]
y = switching_df['next_provider'].astype(int)

le = LabelEncoder()
y_encoded    = le.fit_transform(y)
target_names = [provider_map[float(cls)] for cls in le.classes_]

print(f'피처 수: {len(features)}개')
print(f'클래스: {target_names}')

피처 수: 17개
클래스: ['SKT', 'KT', 'LGU+']


In [18]:
# ----------------------------------
# 6. 학습 / 검증 분할 (8:2)
# ----------------------------------

X_train, X_val, y_train, y_val = train_test_split(
    X, y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print(f'Train: {len(X_train):,}  |  Val: {len(X_val):,}')
print('\n[Train 클래스 분포]')
print(pd.Series(y_train).value_counts().rename(index=dict(enumerate(target_names))))

Train: 33,856  |  Val: 8,464

[Train 클래스 분포]
SKT     13310
KT      10961
LGU+     9585
Name: count, dtype: int64


In [19]:
# ----------------------------------
# 7. 모델 학습
# ----------------------------------

params = {
    'objective'        : 'multiclass',
    'num_class'        : len(le.classes_),
    'metric'           : 'multi_logloss',
    'boosting_type'    : 'gbdt',
    'learning_rate'    : 0.05,
    'num_leaves'       : 15,
    'min_child_samples': 20,
    'subsample'        : 0.8,
    'colsample_bytree' : 0.8,
    'reg_alpha'        : 0.5,
    'reg_lambda'       : 0.5,
    'class_weight'     : 'balanced',
    'random_state'     : 42,
    'verbose'          : -1
}

train_data = lgb.Dataset(X_train, label=y_train)
val_data   = lgb.Dataset(X_val,   label=y_val, reference=train_data)

model = lgb.train(
    params,
    train_data,
    num_boost_round=1000,
    valid_sets=[train_data, val_data],
    callbacks=[
        lgb.early_stopping(stopping_rounds=30, verbose=False),
        lgb.log_evaluation(-1)
    ]
)

print('train finish')

train finish


In [20]:
# ----------------------------------
# 8. 평가
# ----------------------------------

y_val_prob = model.predict(X_val)
y_val_pred = np.argmax(y_val_prob, axis=1)

print('========================================================')
print(f' [모델 평가] 검증 데이터셋 정확도: {accuracy_score(y_val, y_val_pred):.2%}')
print('========================================================')

print('\n[Classification Report]')
print(classification_report(y_val, y_val_pred, target_names=target_names))

print('[Confusion Matrix]')
cm = confusion_matrix(y_val, y_val_pred)
cm_df = pd.DataFrame(
    cm,
    index=[f'실제_{t}'   for t in target_names],
    columns=[f'예측_{t}' for t in target_names]
)
print(cm_df)

 [모델 평가] 검증 데이터셋 정확도: 62.22%

[Classification Report]
              precision    recall  f1-score   support

         SKT       0.66      0.98      0.79      3327
          KT       0.59      0.54      0.57      2741
        LGU+       0.53      0.21      0.30      2396

    accuracy                           0.62      8464
   macro avg       0.59      0.58      0.55      8464
weighted avg       0.60      0.62      0.58      8464

[Confusion Matrix]
         예측_SKT  예측_KT  예측_LGU+
실제_SKT     3266     42       19
실제_KT       820   1490      431
실제_LGU+     892    994      510


In [21]:
# ----------------------------------
# 9. 변수 중요도
# ----------------------------------

importance_df = pd.DataFrame({
    'feature'   : features,
    'importance': model.feature_importance(importance_type='gain')
}).sort_values('importance', ascending=False).reset_index(drop=True)

print('[변수 중요도 Top 17]')
print(importance_df.to_string(index=False))

[변수 중요도 Top 17]
            feature    importance
           provider 294999.854985
               area   6446.318648
   cost_change_rate   5675.920575
 monthly_total_cost   4823.017281
             tenure   3788.519385
monthly_installment   3081.656604
                age   2479.027548
      total_changes   2126.431730
             school   1590.854283
             income   1411.400401
  is_mobile_bundled   1215.453052
           marriage    818.163324
     household_size    800.272939
   provider_changed    757.185743
         cost_payer    457.548499
                job    364.199209
             gender    277.185488


In [23]:
# ----------------------------------
# 10. 최종 결과 출력
# ----------------------------------

pred_probs       = model.predict(X)
pred_indices     = np.argmax(pred_probs, axis=1)
max_probs        = np.max(pred_probs, axis=1)
predicted_labels = le.inverse_transform(pred_indices)

switching_df['현재 통신사']      = switching_df['provider'].map(provider_map)
switching_df['실제 다음 통신사'] = switching_df['next_provider'].map(provider_map)


# 통신사별 확률 컬럼 추가
for i, name in enumerate(target_names):
    switching_df[f'{name} 이동 확률(%)'] = [f'{p[i]*100:.1f}%' for p in pred_probs]

output_cols = (
    ['id', 'year', '현재 통신사'] +
    [f'{t} 확률(%)' for t in target_names] +
    [ '실제 다음 통신사']
)

result_show = switching_df[output_cols].head(10).copy()
result_show.columns = (
    ['고객 ID', '이탈 연도', '현재 통신사'] +
    [f'{t} 확률(%)' for t in target_names] +
    [ '실제 다음 통신사']
)

print('========== 이탈 고객별 차기 통신사 예측 결과 (상위 10개) ==========')
print(result_show.to_string(index=False))

========== 이탈 고객별 차기 통신사 예측 결과 (상위 10개) ==========
 고객 ID  이탈 연도 현재 통신사 SKT 확률(%) KT 확률(%) LGU+ 확률(%) 실제 다음 통신사
 10001     13    SKT      0.0%    82.5%      17.5%        KT
 10001     14     KT     63.6%     0.4%      36.0%      LGU+
 10001     15   LGU+     56.6%    43.3%       0.0%        KT
 10001     16     KT     72.0%     0.3%      27.7%       SKT
 10001     17    SKT      0.0%    61.1%      38.9%        KT
 10001     18     KT     57.8%     0.6%      41.7%       SKT
 10001     19    SKT      0.0%    52.3%      47.7%        KT
 10002     14     KT     75.0%     0.4%      24.6%      LGU+
 10002     15   LGU+     64.9%    35.1%       0.0%       SKT
 10002     19    SKT      0.0%    55.4%      44.6%        KT
